In [2]:
from symbols import i, j, u, v
import sympy as sp
from sympy import IndexedBase, collect
import importlib
# from derive import MomentumFVM,MomentumFVM2, sourceTerm, SecondOrderDiffusion, UDS,CDS, HYBRID, FourthOrderDiffusion
import derive

In [5]:
importlib.reload(derive)
conv_scheme=derive.UDS()
diff_scheme=derive.SecondOrderDiffusion()
u_momentum_1 = derive.MomentumFVM(
    kind="V",
    offset=0,
    convection_scheme=conv_scheme,
    diffusion_scheme=diff_scheme
)
phi = u_momentum_1.phi
flux_cv = u_momentum_1.flux_cv
i = u_momentum_1.i
j = u_momentum_1.j
conv_scheme.build(phi, flux_cv, i, j)

u_momentum_1.eq

-dx*p[i, j + 1] + dx*p[i, j] + (-NU*dy/dx - Max(0.0, -Fe))*phi[i + 1, j] + (-NU*dy/dx - Max(0.0, Fw))*phi[i - 1, j] + (-NU*dx/dy - Max(0.0, -Fn))*phi[i, j + 1] + (-NU*dx/dy - Max(0.0, Fs))*phi[i, j - 1] + (2*NU*dx/dy + 2*NU*dy/dx + Max(0.0, Fe) + Max(0.0, Fn) + Max(0.0, -Fs) + Max(0.0, -Fw))*phi[i, j]

In [4]:
importlib.reload(derive)
conv_scheme=derive.UDS()
diff_scheme=derive.SecondOrderDiffusion()
u_momentum_1 = derive.MomentumFVM(
    kind="U",
    offset=0,
    convection_scheme=conv_scheme,
    diffusion_scheme=diff_scheme
)
phi = u_momentum_1.phi
flux_cv = u_momentum_1.flux_cv
i = u_momentum_1.i
j = u_momentum_1.j
conv_scheme.build(phi, flux_cv, i, j)
u_momentum_1._to_fortran()

u_momentum_1.flux_cv

      ! FACE FLUX FOR U-CV AT (I,J):
      FEU = DY*(0.5*U(I + 1, J) + 0.5*U(I, J))
      FWU = DY*(0.5*U(I - 1, J) + 0.5*U(I, J))
      FNU = DX*(0.5*V(I + 1, J) + 0.5*V(I, J))
      FSU = DX*(0.5*V(I + 1, J - 1) + 0.5*V(I, J - 1))

      ! A COEFFICIENTS FOR U-CV AT (I,J):
      AEU(I,J) = (1/RE)*DY/DX + MAX(0.0, -FEU)
      AEEU(I,J) = 0
      AWU(I,J) = (1/RE)*DY/DX + MAX(0.0, FWU)
      AWWU(I,J) = 0
      ANU(I,J) = (1/RE)*DX/DY + MAX(0.0, -FNU)
      ANNU(I,J) = 0
      ASU(I,J) = (1/RE)*DX/DY + MAX(0.0, FSU)
      ASSU(I,J) = 0
      APU(I,J) = 2*(1/RE)*DX/DY + 2*(1/RE)*DY/DX
     &    + MAX(0.0, FEU) + MAX(0.0, FNU)
     &    + MAX(0.0, -FSU) + MAX(0.0, -FWU)

      ! B RESIDUAL FOR U-CV AT (I,J):
      BU(I,J)=DY*(-P(I + 1, J) + P(I, J))
     &    - APU(I,J)*U(I, J)
     &    + AEU(I,J)*U(I + 1, J)
     &    + AEEU(I,J)*U(I + 2, J)
     &    + AWU(I,J)*U(I - 1, J)
     &    + AWWU(I,J)*U(I - 2, J)
     &    + ANU(I,J)*U(I, J + 1)
     &    + ANNU(I,J)*U(I, J + 2)
     &    + 